In [8]:
import logging

logging.basicConfig(level=logging.DEBUG)

## Microservices

In [3]:
monomorph_results = {
    ("microrefact", "7ep-demo", "manual"): {
        "path": "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data_important/7ep-demo/run2/7ep-demo-2508061819-bce4",
    },
    ("microrefact", "jpetstore-6", "manual"): {
        "path": "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data_important_bck/jpetstore-6/run2/jpetstore-6-2507232035-7d27",
    },
    ("microrefact", "spring-petclinic", "manual"): {
        "path": "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data_important/spring-petclinic/run2/spring-petclinic-2507231543-0147",
    },
}

In [17]:
import shutil, os

buildable_versions = {
    "7ep-demo": "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data_important/7ep-demo/run2/7ep-demo-2508061819-bce4",
    "jpetstore-6": "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data_important_bck/jpetstore-6/run2/jpetstore-6-2507232035-7d27",
    "spring-petclinic": "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data_important/spring-petclinic/run2/spring-petclinic-2507231543-0147",
}

microservices_path = "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/microservices"
microservices_to_modify_path = "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/microservices-with-dockerfiles"
monoliths_with_dockerfiles_path = "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/monoliths-with-dockerfiles"
reports_path = "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/reports"

In [22]:
for app in buildable_versions:
    print(f"Processing {app}...")
    app_path = buildable_versions[app]
    target_app_path1 = f"{microservices_path}/{app}"
    if not os.path.exists(target_app_path1):
        os.makedirs(microservices_to_modify_path, exist_ok=True)
        shutil.copytree(app_path, target_app_path1)
        print(f"Copied {app} microservices to {target_app_path1}")
    else:
        print(f"{app} microservices already exist at {target_app_path1}")
    target_app_path2 = f"{microservices_to_modify_path}/{app}"
    if not os.path.exists(target_app_path2):
        shutil.copytree(app_path, target_app_path2)
        print(f"Copied {app} microservices to {target_app_path2}")
    else:
        print(f"{app} microservices already exist at {target_app_path2}")
    # Remove the debugging directory if it exists
    debugging_path = f"{target_app_path2}/debugging"
    if os.path.exists(debugging_path):
        shutil.rmtree(debugging_path)
        print(f"Removed debugging directory at {debugging_path}")
    else:
        print(f"No debugging directory to remove at {debugging_path}")
    # Copy the Dockerfile to the target path
    source_dockerfile_path = f"{monoliths_with_dockerfiles_path}/{app}/DockerfileMRtesting"
    microservices = [i for i in os.listdir(target_app_path2) if not i.startswith(".") and i!="debugging" and os.path.isdir(os.path.join(target_app_path2, i))]
    for ms in microservices:
        target_dockerfile_path = f"{target_app_path2}/{ms}/DockerfileMRtesting"
        if not os.path.exists(target_dockerfile_path):
            shutil.copy(source_dockerfile_path, target_dockerfile_path)
            print(f"Copied Dockerfile to {target_dockerfile_path}")
        else:
            print(f"Dockerfile already exists at {target_dockerfile_path}")

Processing 7ep-demo...
7ep-demo microservices already exist at /Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/microservices/7ep-demo
7ep-demo microservices already exist at /Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/microservices-with-dockerfiles/7ep-demo
No debugging directory to remove at /Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/microservices-with-dockerfiles/7ep-demo/debugging
Copied Dockerfile to /Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/microservices-with-dockerfiles/7ep-demo/mathematics/DockerfileMRtesting
Copied Dockerfile to /Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/microservices-with-dockerfiles/7ep-demo/library/DockerfileMRtesting
Copied Dockerfile to /Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps

In [1]:
microservice_id_name_map = {
    "7ep-demo": {
        "0": "authentication",
        "1": "library",
        "2": "mathematics",
        "3": "persistence",
    },
    "jpetstore-6": {
        "0": "account",
        "1": "catalog",
        "2": "order",
    },
    "spring-petclinic": {
        "0": "customers",
        "1": "vets",
        "2": "visits",
    }
}

In [2]:
from pathlib import Path
import shutil
import os

from refacteval.testing.docker import test_microservice_with_compose

microservices_to_modify_path = "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/microrefact"
reports_path = "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/reports"

In [29]:
app = "7ep-demo" #"spring-petclinic"#"jpetstore-6" #
server_services = ["authentication"]
ms_app_path = f"{microservices_to_modify_path}/{app}"
microservices = [i for i in os.listdir(ms_app_path) if not i.startswith(".") and i!="debugging" and os.path.isdir(os.path.join(ms_app_path, i))]
print(microservices)

['0', '1', '3', '2']


In [30]:
import re

defined_server_pattern = re.compile(r"\s*([a-zA-Z0-9_-]+-server):\s*")

docker_compose_file = f"{ms_app_path}/docker-compose.yml"

assert os.path.exists(docker_compose_file)

with open(docker_compose_file, "r") as f:
    docker_compose_content = f.read()

defined_server_services = list(set(defined_server_pattern.findall(docker_compose_content)))
defined_server_services

[]

In [32]:
from subprocess import CalledProcessError

for ms in microservices:
    print(f"Testing {ms}...")
    ms_path = f"{ms_app_path}/{ms}"
    ms_report_path = Path(f"{reports_path}/microrefact/{app}")
    if (ms_report_path/ms).exists():
        print(f"Report for {ms} already exists at {ms_report_path/ms}")
        continue
    else:
        print(f"Creating report directory for {ms} at {ms_report_path/ms}")
    if not os.path.exists(ms_report_path):
        os.makedirs(ms_report_path, exist_ok=True)
    build_tool = "gradle" if app=="7ep-demo" else "maven"
    try:
        test_microservice_with_compose(ms, defined_server_services, Path(ms_app_path), ms_report_path, build_tool, True, True,
                                       test_runner_service=f"{microservice_id_name_map[app][ms]}-tests")
    except CalledProcessError as e:
        print(f"Testing {ms} failed with error: {e}")
        continue

INFO:refacteval:--- Starting test run for microservice: 1 ---
INFO:refacteval:Working directory: /Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/microrefact/7ep-demo
INFO:refacteval:Reports will be saved to: /Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/reports/microrefact/7ep-demo/1
INFO:refacteval:No server services to start.
INFO:refacteval:Running test container for service: 'library-tests'


Testing 0...
Report for 0 already exists at /Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/reports/microrefact/7ep-demo/0
Testing 1...
Creating report directory for 1 at /Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/reports/microrefact/7ep-demo/1


time="2025-12-08T16:52:59-05:00" level=warning msg="/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/microrefact/7ep-demo/docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
 Network 7ep-demo_testing_network  Creating
 Network 7ep-demo_testing_network  Created
time="2025-12-08T16:53:00-05:00" level=warning msg="Found orphan containers ([authentication-tests-cont-1765230554]) for this project. If you removed or renamed this service in your compose file, you can run this command with the --remove-orphans flag to clean it up."


#1 [internal] load local bake definitions
#1 reading from stdin 709B done
#1 DONE 0.0s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 1.15kB done
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/library/gradle:7.4.2-jdk11-jammy
#3 DONE 0.0s

#4 [internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [build 1/9] FROM docker.io/library/gradle:7.4.2-jdk11-jammy
#5 DONE 0.0s

#6 [internal] load build context
#6 transferring context: 290.15kB 0.0s done
#6 DONE 0.0s

#7 [build 3/9] COPY build.gradle settings.gradle* ./
#7 CACHED

#8 [build 2/9] WORKDIR /app
#8 CACHED

#9 [build 4/9] COPY gradle ./gradle
#9 CACHED

#10 [build 5/9] COPY gradle* ./
#10 CACHED

#11 [build 6/9] COPY src/main/webapp ./src/main/webapp
#11 CACHED

#12 [build 7/9] RUN gradle dependencies --no-daemon
#12 0.445 
#12 0.445 Welcome to Gradle 7.4.2!
#12 0.445 
#12 0.445 Here are the highlights of this release:
#12 0.445  - Aggregated test and JaCoCo reports


Note: /app/src/test/java/com/coveros/training/library/LibraryUtilsTests.java uses or overrides a deprecated API.
Note: Recompile with -Xlint:deprecation for details.


...

Testing 3 failed with error: Command '['docker-compose', 'run', '--name', 'persistence-tests-cont-1765230910', 'persistence-tests']' returned non-zero exit status 1.
Testing 2...
Creating report directory for 2 at /Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/reports/microrefact/7ep-demo/2
#1 [internal] load local bake definitions
#1 reading from stdin 725B done
#1 DONE 0.0s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 1.15kB done
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/library/gradle:7.4.2-jdk11-jammy
#3 DONE 0.0s

#4 [internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [build 1/9] FROM docker.io/library/gradle:7.4.2-jdk11-jammy
#5 DONE 0.0s

#6 [internal] load build context
#6 transferring context: 209.79kB 0.0s done
#6 DONE 0.0s

#7 [build 5/9] COPY gradle* ./
#7 CACHED

#8 [build 4/9] COPY gradle ./gradle
#8 CACHED

#9 [build 2/9] WORKDIR /app
#9 CACHED

#10 [


73 tests completed, 5 failed, 6 skipped



> Task :jacocoTestReport
Cached resource https://repo.maven.apache.org/maven2/org/ow2/asm/asm-commons/9.1/asm-commons-9.1.pom is up-to-date (lastModified: Sat Feb 06 12:31:13 UTC 2021).
Cached resource https://repo.maven.apache.org/maven2/org/ow2/asm/asm-tree/9.1/asm-tree-9.1.pom is up-to-date (lastModified: Sat Feb 06 12:31:19 UTC 2021).
Excluding []
Caching disabled for task ':jacocoTestReport' because:
  Build cache is disabled
Task ':jacocoTestReport' is not up-to-date because:
  No history is available.
[ant:jacocoReport] Loading execution data file /app/build/jacoco/test.exec
[ant:jacocoReport] Writing bundle 'app' with 28 classes
:jacocoTestReport (Thread[included builds,5,main]) completed. Took 0.857 secs.

Deprecated Gradle features were used in this build, making it incompatible with Gradle 8.0.

You can use '--warning-mode all' to show the individual deprecation warnings and determine if they come from your own scripts or plugins.

See https://docs.gradle.org/7.4.2/userguid


FAILURE: Build failed with an exception.

* What went wrong:
Execution failed for task ':test'.
> There were failing tests. See the report at: file:///app/build/reports/tests/test/index.html

* Try:
> Run with --stacktrace option to get the stack trace.
> Run with --debug option to get more log output.
> Run with --scan to get full insights.

* Get more help at https://help.gradle.org

BUILD FAILED in 9s

ERROR:refacteval:Test run failed for 'mathematics-tests' with exit code 1.
INFO:refacteval:Entering cleanup and report extraction phase...
INFO:refacteval:Gradle test reports copied to /var/folders/rx/vycj7vys6v78_nd6qmwm7nj40000gn/T/tmpeatzb78c
INFO:refacteval:Output directory /Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/reports/microrefact/7ep-demo/2 exists but is empty. Proceeding to copy reports.
INFO:refacteval:Test reports copied to /Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/reports/mi

Testing 2 failed with error: Command '['docker-compose', 'run', '--name', 'mathematics-tests-cont-1765231028', 'mathematics-tests']' returned non-zero exit status 1.


In [33]:
5

5

## Parse the reports

### Monoliths

In [46]:
app = "spring-petclinic" #"jpetstore-6"  #"7ep-demo"
reports_path = "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/reports"
app_report_path = Path(f"{reports_path}/monoliths/{app}")

In [35]:
from refacteval.testing.report import ReportParser

In [48]:
parser = ReportParser()

report = parser.extract(app_report_path)

Skipping testTriggerException


In [53]:
report['org.springframework.samples.petclinic.owner.OwnerControllerTests']['test_cases'].keys()

dict_keys(['testProcessUpdateOwnerFormHasErrors', 'testProcessCreationFormSuccess', 'testInitFindForm', 'testShowOwner', 'testInitUpdateOwnerForm', 'testInitCreationForm', 'testProcessFindFormByLastName', 'testProcessFindFormNoOwnersFound', 'testProcessFindFormSuccess', 'testProcessUpdateOwnerFormSuccess', 'testProcessCreationFormHasErrors', 'testProcessUpdateOwnerFormUnchangedSuccess'])

In [34]:
import pandas as pd

INFO:numexpr.utils:NumExpr defaulting to 10 threads.


In [73]:
columns = ["app", "ms_name_or_monolith", "test_class", "test_method", "status", "time"]
reports_path = "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/reports"

rows = []
rows_per_app = {}

for app in ["spring-petclinic", "jpetstore-6", "7ep-demo"]:
    app_report_path = Path(f"{reports_path}/monoliths/{app}")
    print(f"Testing {app}...")
    rows_per_app[app] = []
    parser = ReportParser()
    report = parser.extract(app_report_path)
    print(f"Report for {app}:")
    for test_class in report:
        print(f"Test class: {test_class}")
        for test_method in report[test_class]['test_cases']:
            print(f"  Test method: {test_method}")
            print(f"    Status: {report[test_class]['test_cases'][test_method]['status']}")
            print(f"    Time: {report[test_class]['test_cases'][test_method]['time']}")
            rows.append({
                "app": app,
                "ms_name_or_monolith": "monolith",
                "test_class": test_class,
                "test_method": test_method,
                "status": report[test_class]['test_cases'][test_method]['status'],
                "time": report[test_class]['test_cases'][test_method]['time'],
            })
            rows_per_app[app].append(report[test_class]['test_cases'][test_method]['status'])
        print()
    print("="*100)
    print()

df_mono = pd.DataFrame(rows, columns=columns)

Skipping testTriggerException
Skipping testShouldPerformWell
Skipping testShouldBe16Years3Months_HappyPath
Skipping testShouldReturnEvenTime
Skipping testShouldMockOutsideMethodsPart2
Skipping testShouldGetPairResult
Skipping testShouldAddTwoDecimals
Skipping testShouldAddTwoIntegers
Skipping testShouldMockOutsideMethods
Skipping testShouldGetStringVersionOfResult
Skipping testShouldHaveSufficientEntropyInPassword
Skipping testShouldHaveInsufficientEntropyInPassword
Skipping testShouldHaveSuccessfulEntropyInPassword


Testing spring-petclinic...
Report for spring-petclinic:
Test class: org.springframework.samples.petclinic.owner.OwnerControllerTests
  Test method: testProcessUpdateOwnerFormHasErrors
    Status: passed
    Time: 0.35
  Test method: testProcessCreationFormSuccess
    Status: passed
    Time: 0.006
  Test method: testInitFindForm
    Status: passed
    Time: 0.01
  Test method: testShowOwner
    Status: passed
    Time: 0.031
  Test method: testInitUpdateOwnerForm
    Status: passed
    Time: 0.012
  Test method: testInitCreationForm
    Status: passed
    Time: 0.01
  Test method: testProcessFindFormByLastName
    Status: passed
    Time: 0.005
  Test method: testProcessFindFormNoOwnersFound
    Status: passed
    Time: 0.007
  Test method: testProcessFindFormSuccess
    Status: passed
    Time: 0.014
  Test method: testProcessUpdateOwnerFormSuccess
    Status: passed
    Time: 0.004
  Test method: testProcessCreationFormHasErrors
    Status: passed
    Time: 0.008
  Test method: test

In [61]:
df_mono.shape

(322, 6)

In [63]:
df_mono

,app,ms_name_or_monolith,test_class,test_method,status,time
0,spring-petclinic,monolith,org.springframework.samples.petclinic.owner.Ow...,testProcessUpdateOwnerFormHasErrors,passed,0.350
1,spring-petclinic,monolith,org.springframework.samples.petclinic.owner.Ow...,testProcessCreationFormSuccess,passed,0.006
2,spring-petclinic,monolith,org.springframework.samples.petclinic.owner.Ow...,testInitFindForm,passed,0.010
3,spring-petclinic,monolith,org.springframework.samples.petclinic.owner.Ow...,testShowOwner,passed,0.031
4,spring-petclinic,monolith,org.springframework.samples.petclinic.owner.Ow...,testInitUpdateOwnerForm,passed,0.012
...,...,...,...,...,...,...
317,7ep-demo,monolith,com.coveros.training.mathematics.FibonacciTests,testLargerValuesFibAlgo2,passed,0.001
318,7ep-demo,monolith,com.coveros.training.mathematics.FibonacciTests,testSmallValuesFibAlgo1,passed,0.000
319,7ep-demo,monolith,com.coveros.training.mathematics.FibonacciTests,testSmallValuesFibAlgo2,passed,0.000
320,7ep-demo,monolith,com.coveros.training.mathematics.FibonacciTests,testLargeValuesFibAlgo1,passed,0.000


### MicroRefact

In [36]:
columns = ["app", "ms_name_or_monolith", "test_class", "test_method", "status", "time"]
reports_path = "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/reports"

rows = []
rows_per_ms = {}

for app in ["spring-petclinic", "jpetstore-6", "7ep-demo"]:
    app_report_path = Path(f"{reports_path}/microrefact/{app}")
    print(f"Testing {app}...")
    rows_per_ms[app] = {}
    microservices = [i for i in os.listdir(app_report_path) if not i.startswith(".") and os.path.isdir(os.path.join(app_report_path, i))]
    for ms in microservices:
        rows_per_ms[app][ms] = []
        parser = ReportParser()
        report = parser.extract(app_report_path/ms)
        print(f"Report for {app}/{ms}:")
        for test_class in report:
            # print(f"Test class: {test_class}")
            for test_method in report[test_class]['test_cases']:
                # print(f"  Test method: {test_method}")
                # print(f"    Status: {report[test_class]['test_cases'][test_method]['status']}")
                # print(f"    Time: {report[test_class]['test_cases'][test_method]['time']}")
                rows.append({
                    "app": app,
                    "ms_name_or_monolith": ms,
                    "test_class": test_class,
                    "test_method": test_method,
                    "status": report[test_class]['test_cases'][test_method]['status'],
                    "time": report[test_class]['test_cases'][test_method]['time'],
                })
                rows_per_ms[app][ms].append(report[test_class]['test_cases'][test_method]['status'])
            # print()
        print("="*100)
        print()
    print("="*100)
    print()

df_microrefact = pd.DataFrame(rows, columns=columns)

INFO:refacteval:Parsing report TEST-org.springframework.samples.petclinic.owner.OwnerControllerTests.xml for class org.springframework.samples.petclinic.owner.OwnerControllerTests
INFO:refacteval:Parsing report TEST-org.springframework.samples.petclinic.owner.PetControllerTests.xml for class org.springframework.samples.petclinic.owner.PetControllerTests
INFO:refacteval:Parsing report TEST-org.springframework.samples.petclinic.model.ValidatorTests.xml for class org.springframework.samples.petclinic.model.ValidatorTests
INFO:refacteval:Parsing report TEST-org.springframework.samples.petclinic.service.ClinicServiceTests.xml for class org.springframework.samples.petclinic.service.ClinicServiceTests
INFO:refacteval:Parsing report TEST-org.springframework.samples.petclinic.system.CrashControllerTests.xml for class org.springframework.samples.petclinic.system.CrashControllerTests
INFO:refacteval:Parsing report TEST-org.springframework.samples.petclinic.owner.PetTypeFormatterTests.xml for clas

Testing spring-petclinic...
Report for spring-petclinic/0:

Report for spring-petclinic/1:

Report for spring-petclinic/2:


Testing jpetstore-6...
Report for jpetstore-6/0:

Report for jpetstore-6/1:

Report for jpetstore-6/2:


Testing 7ep-demo...
Report for 7ep-demo/0:

Report for 7ep-demo/1:

Report for 7ep-demo/3:

Report for 7ep-demo/2:




In [37]:
df_microrefact.shape

(387, 6)

In [75]:
for app in rows_per_app:
    print(f"App: {app}: {len(rows_per_app[app])}")

App: spring-petclinic: 41
App: jpetstore-6: 74
App: 7ep-demo: 207


In [40]:
for app in rows_per_ms:
    sum = 0
    for ms in rows_per_ms[app]:
        print(f"App: {app}/{ms}: {len(rows_per_ms[app][ms])}")
        sum += len(rows_per_ms[app][ms])
    print(f"App: {app}/all: {sum}")
    print()

App: spring-petclinic/0: 33
App: spring-petclinic/1: 15
App: spring-petclinic/2: 5
App: spring-petclinic/all: 53

App: jpetstore-6/0: 18
App: jpetstore-6/1: 29
App: jpetstore-6/2: 27
App: jpetstore-6/all: 74

App: 7ep-demo/0: 61
App: 7ep-demo/1: 94
App: 7ep-demo/3: 32
App: 7ep-demo/2: 73
App: 7ep-demo/all: 260



In [41]:
save_path = f"{reports_path}/parsed_reports_microrefact.csv"
df_microrefact.to_csv(save_path, index=False)

## Analyze the results

In [42]:
import pandas as pd

monolith_path = "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/reports/parsed_reports_monoliths.csv"
microservices_path = "/Users/khalsel/Documents/projects/MonoRefact/MonoRefact/data/refacteval-final/testing-apps/reports/parsed_reports_microrefact.csv"

df_mono = pd.read_csv(monolith_path)
df_microrefact = pd.read_csv(microservices_path)

In [43]:
mono_tests = df_mono.apply(lambda row: f"{row['app']}::{row['test_class']}::{row['test_method']}", axis=1).tolist()
df_mono_indexed = df_mono.copy()
df_mono_indexed["test_fqn"] = mono_tests
df_mono_indexed.set_index("test_fqn", inplace=True)
mm_tests = df_microrefact.apply(lambda row: f"{row['app']}::{row['test_class']}::{row['test_method']}", axis=1).tolist()
df_microrefact["test_fqn"] = mm_tests

In [44]:
rows = []

for app in df_microrefact["app"].unique():
    dft = df_microrefact[df_microrefact.app==app]
    for ms in dft["ms_name_or_monolith"].unique():
        dft2 = dft[(dft.ms_name_or_monolith==ms)]
        count = len(dft2)
        mono_dft2 = df_mono_indexed.loc[dft2["test_fqn"]]
        matching_tests_index = dft2.apply(lambda row: row["status"]==mono_dft2.loc[row["test_fqn"], "status"], axis=1)
        matching_tests_count = matching_tests_index.sum()
        passed_tests = dft2[dft2.status == "passed"]
        mono_passed_tests = df_mono_indexed.loc[passed_tests["test_fqn"]]
        passed_count = len(passed_tests)
        total_test_count = len(dft2)
        total_time = passed_tests["time"].sum()
        avg_time = passed_tests["time"].mean() if passed_count > 0 else 0
        mono_total_time = mono_passed_tests["time"].sum()
        mono_avg_time = mono_passed_tests["time"].mean() if passed_count > 0 else 0
        rows.append({
            "app": app,
            "ms_name_or_monolith": ms,
            "matching_tests_count": matching_tests_count,
            "passed_tests_count": passed_count,
            "total_test_count": total_test_count,
            "fpr": matching_tests_count / total_test_count if total_test_count > 0 else 0,
            "passed_total_time": total_time,
            "passed_avg_time": avg_time,
            "passed_mono_total_time": mono_total_time,
            "passed_mono_avg_time": mono_avg_time,
        })
df_summary = pd.DataFrame(rows, columns=["app", "ms_name_or_monolith", "matching_tests_count", "passed_tests_count", "total_test_count", "fpr", "passed_total_time", "passed_avg_time", "passed_mono_total_time", "passed_mono_avg_time"])
df_summary

,app,ms_name_or_monolith,matching_tests_count,passed_tests_count,total_test_count,fpr,passed_total_time,passed_avg_time,passed_mono_total_time,passed_mono_avg_time
0,spring-petclinic,0,23,22,33,0.696970,1.072,0.048727,0.542,0.024636
1,spring-petclinic,1,5,4,15,0.333333,0.841,0.210250,0.079,0.019750
2,spring-petclinic,2,2,1,5,0.400000,0.392,0.392000,0.003,0.003000
3,jpetstore-6,0,15,15,18,0.833333,0.878,0.058533,0.737,0.049133
4,jpetstore-6,1,29,29,29,1.000000,1.041,0.035897,0.092,0.003172
5,jpetstore-6,2,16,16,27,0.592593,0.952,0.059500,0.204,0.012750
6,7ep-demo,0,50,46,61,0.819672,0.609,0.013239,0.533,0.011587
7,7ep-demo,1,36,36,94,0.382979,0.167,0.004639,0.098,0.002722
8,7ep-demo,3,27,27,32,0.843750,0.137,0.005074,0.072,0.002667
9,7ep-demo,2,68,62,73,0.931507,0.197,0.003177,0.138,0.002226


In [45]:
df_summary = df_summary[["app", "ms_name_or_monolith", "matching_tests_count", "passed_tests_count", "total_test_count", "fpr"]]
overall_rows = []
for app in df_summary["app"].unique():
    dft = df_summary[df_summary.app==app]
    overall_rows.append({
        "app": app,
        "ms_name_or_monolith": "Overall",
        "matching_tests_count": dft["matching_tests_count"].sum(),
        "passed_tests_count": dft["passed_tests_count"].sum(),
        "total_test_count": dft["total_test_count"].sum(),
        "fpr": dft["matching_tests_count"].sum() / dft["total_test_count"].sum() if dft["total_test_count"].sum() > 0 else 0,
    })
all_overall_row = {
    "app": "Overall",
    "ms_name_or_monolith": "Overall",
    "matching_tests_count": df_summary["matching_tests_count"].sum(),
    "passed_tests_count": df_summary["passed_tests_count"].sum(),
    "total_test_count": df_summary["total_test_count"].sum(),
    "fpr": df_summary["matching_tests_count"].sum() / df_summary["total_test_count"].sum() if df_summary["total_test_count"].sum() > 0 else 0,
}
overall_rows.append(all_overall_row)
df_overall_summary = pd.concat([
    df_summary,
    pd.DataFrame(overall_rows, columns=df_summary.columns),
], ignore_index=True)
# sort by app and ms_name_or_monolith but the overall row should be at the end of each app and then the overall row at the end
df_overall_summary["app"] = df_overall_summary["app"].astype(str)
df_overall_summary["ms_name_or_monolith"] = df_overall_summary["ms_name_or_monolith"].astype(str)
df_overall_summary["app"] = df_overall_summary["app"].apply(lambda x: x if x != "Overall" else "zZZ_Overall")
df_overall_summary["ms_name_or_monolith"] = df_overall_summary["ms_name_or_monolith"].apply(lambda x: x if x != "Overall" else "zZZ_Overall")
df_overall_summary = df_overall_summary.sort_values(["app", "ms_name_or_monolith"])
df_overall_summary["app"] = df_overall_summary["app"].apply(lambda x: x if x != "zZZ_Overall" else "Overall")
df_overall_summary["ms_name_or_monolith"] = df_overall_summary["ms_name_or_monolith"].apply(lambda x: x if x != "zZZ_Overall" else "Overall")
# sort by app
df_overall_summary

,app,ms_name_or_monolith,matching_tests_count,passed_tests_count,total_test_count,fpr
6,7ep-demo,0,50,46,61,0.819672
7,7ep-demo,1,36,36,94,0.382979
9,7ep-demo,2,68,62,73,0.931507
8,7ep-demo,3,27,27,32,0.843750
12,7ep-demo,Overall,181,171,260,0.696154
3,jpetstore-6,0,15,15,18,0.833333
4,jpetstore-6,1,29,29,29,1.000000
5,jpetstore-6,2,16,16,27,0.592593
11,jpetstore-6,Overall,60,60,74,0.810811
0,spring-petclinic,0,23,22,33,0.696970


In [48]:
# Format the DataFrame to display as a table in latex

df_overall_summary_latex = df_overall_summary.copy()
df_overall_summary_latex = df_overall_summary_latex.rename(columns={
    "matching_tests_count": "Matching Tests",
    "passed_tests_count": "Passed Tests",
    "total_test_count": "Total Tests",
    "fpr": "FPR",
    "ms_name_or_monolith": "Microservice",
    "app": "Application",
})
df_overall_summary_latex["Microservice"] = df_overall_summary_latex.apply(lambda x: microservice_id_name_map[x["Application"]][x["Microservice"]] if x["Microservice"] != "Overall" else x["Microservice"], axis=1)
df_overall_summary_latex.replace(
    {
        'Application': {
            'spring-petclinic': 'PetClinic',
            'jpetstore-6': 'JPetStore',
            '7ep-demo': '7ep Demo',
        }
    },
    inplace=True
)
df_overall_summary_latex["Microservice"] = df_overall_summary_latex["Microservice"].apply(lambda x: x.title())
df_overall_summary_latex = df_overall_summary_latex.set_index(['Application', 'Microservice'])
df_overall_summary_latex["FPR"] = df_overall_summary_latex["FPR"].apply(lambda x: f"{x*100:.1f}\\%")
df_overall_summary_latex = df_overall_summary_latex.style.format({
    'Matching Tests': '{:,.0f}',
    'Passed Tests': '{:,.0f}',
    'Total Tests': '{:,.0f}',
    # 'FPR': '{:.1%}',
})
df_overall_summary_latex

In [50]:
df_overall_summary_latex_str = df_overall_summary_latex.to_latex(
    hrules=True,
    label="tab-mm:test_results_microrefact",
    caption="Test Outcome Equivalence Results for MicroRefact Microservices",
    position="htbp",
    column_format="ll|cccc",
)
print(df_overall_summary_latex_str)

\begin{table}[htbp]
\caption{Test Outcome Equivalence Results for MicroRefact Microservices}
\label{tab-mm:test_results_microrefact}
\begin{tabular}{ll|cccc}
\toprule
 &  & Matching Tests & Passed Tests & Total Tests & FPR \\
Application & Microservice &  &  &  &  \\
\midrule
\multirow[c]{5}{*}{7ep Demo} & Authentication & 50 & 46 & 61 & 82.0\% \\
 & Library & 36 & 36 & 94 & 38.3\% \\
 & Mathematics & 68 & 62 & 73 & 93.2\% \\
 & Persistence & 27 & 27 & 32 & 84.4\% \\
 & Overall & 181 & 171 & 260 & 69.6\% \\
\multirow[c]{4}{*}{JPetStore} & Account & 15 & 15 & 18 & 83.3\% \\
 & Catalog & 29 & 29 & 29 & 100.0\% \\
 & Order & 16 & 16 & 27 & 59.3\% \\
 & Overall & 60 & 60 & 74 & 81.1\% \\
\multirow[c]{4}{*}{PetClinic} & Customers & 23 & 22 & 33 & 69.7\% \\
 & Vets & 5 & 4 & 15 & 33.3\% \\
 & Visits & 2 & 1 & 5 & 40.0\% \\
 & Overall & 30 & 27 & 53 & 56.6\% \\
Overall & Overall & 271 & 258 & 387 & 70.0\% \\
\bottomrule
\end{tabular}
\end{table}



In [22]:
failed_rows = []

for app in df_microrefact["app"].unique():
    dft = df_microrefact[df_microrefact.app==app]
    for ms in dft["ms_name_or_monolith"].unique():
        dft2 = dft[(dft.ms_name_or_monolith==ms)]
        count = len(dft2)
        mono_dft2 = df_mono_indexed.loc[dft2["test_fqn"]]
        non_matching_tests_index = ~dft2.apply(lambda row: row["status"]==mono_dft2.loc[row["test_fqn"], "status"], axis=1)
        failed_rows.append(dft2[non_matching_tests_index])
df_non_matching = pd.concat(failed_rows, ignore_index=True)
df_non_matching

,app,ms_name_or_monolith,test_class,test_method,status,time,test_fqn
0,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldFindVets,errored,0.000,spring-petclinic::org.springframework.samples....
1,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldFindOwnersByLastName,errored,0.001,spring-petclinic::org.springframework.samples....
2,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldAddNewVisitForPet,errored,0.000,spring-petclinic::org.springframework.samples....
3,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldUpdateOwner,errored,0.001,spring-petclinic::org.springframework.samples....
4,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldFindVisitsByPetId,errored,0.001,spring-petclinic::org.springframework.samples....
5,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldInsertPetIntoDatabaseAndGenerateId,errored,0.001,spring-petclinic::org.springframework.samples....
6,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldInsertOwner,errored,0.000,spring-petclinic::org.springframework.samples....
7,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldFindSingleOwnerWithPet,errored,0.001,spring-petclinic::org.springframework.samples....
8,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldUpdatePetName,errored,0.001,spring-petclinic::org.springframework.samples....
9,spring-petclinic,customers,org.springframework.samples.petclinic.service....,shouldFindAllPetTypes,errored,0.000,spring-petclinic::org.springframework.samples....


In [23]:
df_non_matching["test_class"].unique()

array(['org.springframework.samples.petclinic.service.ClinicServiceTests',
       'org.springframework.samples.petclinic.owner.VisitControllerTests'],
      dtype=object)